In [22]:
import pandas as pd
import os
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, Model
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import joblib

In [8]:
def readDataInPanda(file_paths):
    folder_path = 'data'
    dataframes = []
    for file_path in file_paths: 
        print("read: " + file_path)
        df = pd.read_csv(file_path)
        dataframes.append(df)
        pandas_data = pd.concat(dataframes, ignore_index=True)

    return pandas_data

In [9]:
# 4. Define Anomaly Detection Function
def isAnomalous(df_d):
    print(df_d)
    anomolous = []
    for idx, row in df_d.iterrows():
        if (
            row["circ.bats[0].state"] == 2 or
            row["circ.bats[1].state"] == 2 or
            row["circ.bats[2].state"] == 2 or
            row["circ.s1[0].state"]  == 2 or
            row["circ.s1[1].state"]  == 2 or
            row["circ.s1[2].state"]  == 2
        ):
            anomolous.append(True)
        else:
            anomolous.append(False)
    return anomolous

In [10]:
#Training data were only the Resistance varies the target voltage is set to 90
def getTrainingDataResistance():
    file_paths = ["./data/training/Circuit_ScenarioDataGeneration_Restistance_res.csv"]
    return readDataInPanda(file_paths)

In [11]:
#Training data were the Resistance and the target voltage changes. Values range from 1. to 100. 
def getTrainingDataResistanceAndVoltage():
    file_paths = ["./data/training/Circuit_ScenarioDataGeneration_Volatage_Restistance_res.csv"]
    return readDataInPanda(file_paths)

In [14]:
def getFaultyData():
    folder_path = "./data/faulty"
    file_paths = [os.path.join(folder_path, f) for f in os.listdir(folder_path) if f.endswith('.csv')]
    return readDataInPanda(file_paths)

In [19]:
def getFaultlessData():
    folder_path = "./data/faultless"
    file_paths = [os.path.join(folder_path, f) for f in os.listdir(folder_path) if f.endswith('.csv')]
    return readDataInPanda(file_paths)

In [20]:
#def loadModelAndScalar(model_name):
    

read: ./data/faultless\Circuit_Scenario1_res.csv
read: ./data/faultless\Circuit_Scenario2_res.csv
read: ./data/faultless\Circuit_Scenario3_res.csv
read: ./data/faultless\Circuit_Scenario4_res.csv


,time,circ1.bats[1].v,circ1.bats[2].v,circ1.bats[3].v,circ1.bats[1].vr,circ1.bats[2].vr,circ1.bats[3].vr,circ1.gnd.p.i,circ1.gnd.p.v,circ1.r_load.i,...,circ1.s[3].m.i,circ1.s[1].m.v,circ1.s[2].m.v,circ1.s[3].m.v,circ1.s[1].p.i,circ1.s[2].p.i,circ1.s[3].p.i,circ1.s[1].p.v,circ1.s[2].p.v,circ1.s[3].p.v
0,0.000000e+00,83.333336,99.999998,99.999998,-16.666664,-0.000002,-0.000002,0.000000e+00,0,0.166667,...,-1.666666e-08,0,0,0,0.166667,1.666666e-08,1.666666e-08,1.666666e-10,1.666666e+01,16.666662
1,1.004082e-10,83.333336,99.999998,99.999998,-16.666664,-0.000002,-0.000002,0.000000e+00,0,0.166667,...,-1.666666e-08,0,0,0,0.166667,1.666666e-08,1.666666e-08,1.666666e-10,1.666666e+01,16.666662
2,1.004082e-10,83.333336,99.999998,99.999998,-16.666664,-0.000002,-0.000002,0.000000e+00,0,0.166667,...,-1.666666e-08,0,0,0,0.166667,1.666666e-08,1.666666e-08,1.666666e-10,1.666666e+01,16.666662
3,1.000000e-01,83.333336,99.999998,99.999998,-16.666664,-0.000002,-0.000002,0.000000e+00,0,0.166667,...,-1.666666e-08,0,0,0,0.166667,1.666666e-08,1.666666e-08,1.666666e-10,1.666666e+01,16.666662
4,2.000000e-01,83.333336,99.999998,99.999998,-16.666664,-0.000002,-0.000002,0.000000e+00,0,0.166667,...,-1.666666e-08,0,0,0,0.166667,1.666666e-08,1.666666e-08,1.666666e-10,1.666666e+01,16.666662
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
571,2.970000e+01,87.179488,87.179488,99.999999,-12.820512,-12.820512,-0.000001,2.775558e-17,0,0.256410,...,-1.282051e-08,0,0,0,0.128205,1.282051e-01,1.282051e-08,1.282051e-10,1.282051e-10,12.820511
572,2.980000e+01,87.179488,87.179488,99.999999,-12.820512,-12.820512,-0.000001,-2.775558e-17,0,0.256410,...,-1.282051e-08,0,0,0,0.128205,1.282051e-01,1.282051e-08,1.282051e-10,1.282051e-10,12.820511
573,2.990000e+01,87.179488,87.179488,99.999999,-12.820512,-12.820512,-0.000001,2.775558e-17,0,0.256410,...,-1.282051e-08,0,0,0,0.128205,1.282051e-01,1.282051e-08,1.282051e-10,1.282051e-10,12.820511
574,3.000000e+01,87.179488,87.179488,99.999999,-12.820512,-12.820512,-0.000001,-2.775558e-17,0,0.256410,...,-1.282051e-08,0,0,0,0.128205,1.282051e-01,1.282051e-08,1.282051e-10,1.282051e-10,12.820511


In [28]:
def getAnomalyMask(df_data): 
    anomolous = []
    for idx, row in df_data.iterrows():
        #print(row)
        # Check any of the specified columns for value == 2
        if (row["circ1.bats[1].state"] != 1 or
            row["circ1.bats[2].state"] != 1 or
            row["circ1.bats[3].state"] != 1 or
            row["circ1.s[1].state"]  != 1 or
            row["circ1.s[2].state"]  != 1 or
            row["circ1.s[3].state"]  != 1):
            anomolous.append(True)
        else:
            anomolous.append(False)
    return np.array(anomolous)

read: ./data/faulty\Circuit_Scenario1_batery_1_broken_res.csv
read: ./data/faulty\Circuit_Scenario1_switch1_broken_res.csv
read: ./data/faulty\Circuit_Scenario2_battery2_broken_res.csv
read: ./data/faulty\Circuit_Scenario2_battery2_empty_res.csv
read: ./data/faulty\Circuit_Scenario2_switch_2_broken_res.csv
read: ./data/faulty\Circuit_Scenario3_battery_1_empty_res.csv
read: ./data/faulty\Circuit_Scenario3_switch_1_broken_res.csv
read: ./data/faulty\Circuit_Scenario4_battery_1_broken_res.csv
read: ./data/faulty\Circuit_Scenario4_battery_1_empty_res.csv
read: ./data/faulty\Circuit_Scenario4_switch_1_broken_res.csv
read: ./data/faultless\Circuit_Scenario1_res.csv
read: ./data/faultless\Circuit_Scenario2_res.csv
read: ./data/faultless\Circuit_Scenario3_res.csv
read: ./data/faultless\Circuit_Scenario4_res.csv


In [ ]:
# Quick test for anomly mask and check data Integrity
anom_mask = getAnomalyMask(getFaultyData())
for anom in anom_mask:
   assert anom == True
anom_mask = getAnomalyMask(getFaultlessData())
for anom in anom_mask:
   assert anom == False